In [1]:
import sys
import os
current_path = os.path.abspath('')
data_path = '/'.join(current_path.split('/')[:-1]) + '/data/'
script_path = '/'.join(current_path.split('/')[:-1]) + '/scripts/'
sys.path.append(script_path)
sys.path.append('/home/fkunneman/.local/lib/python3.10/site-packages/')
sys.path.append('/usr/lib/python3/dist-packages/')

In [2]:
import torch
import gc
import importlib
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface.llms import HuggingFacePipeline
from langchain_community.vectorstores import Chroma
import chromadb
from chromadb.api.types import EmbeddingFunction, Documents, Embeddings

import instruction_agent

In [3]:
### Load models for chatbot and rag in memory
with open('/home/fkunneman/hf.txt') as fin:
    hf = fin.read().strip()
os.environ["HF_TOKEN"] = hf
torch.cuda.empty_cache()
gc.collect()
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained('BramVanroy/GEITje-7B-ultra', use_fast=True)
tokenizer.pad_token = tokenizer.eos_token # Ensure padding token is set
# Initialize language model
chatmodel = AutoModelForCausalLM.from_pretrained('BramVanroy/GEITje-7B-ultra', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")
#chatmodel = AutoModelForCausalLM.from_pretrained('robinsmits/Qwen1.5-7B-Dutch-Chat', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")     
# set pipeline
pipe = pipeline(
    "text-generation",
    model=chatmodel,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=250, # Limit the number of generated tokens for concise responses
    do_sample=True, # Enable sampling for more varied responses
    temperature=0.3 # Control creativity
)
llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [4]:
chroma_client = chromadb.Client()
#embedding_model = SentenceTransformer('jegormeister/bert-base-dutch-cased')
collection = chroma_client.create_collection(
    name="instruction_db",
    configuration={
        "hnsw": {
            "space": "cosine",
            "ef_construction": 30
        }
    }
)

In [20]:

instructions_travel = data_path + 'opslag_inclusieve_spraakassistent_project/instructions_ov_stripped.csv'
instructions_passport = data_path + 'opslag_inclusieve_spraakassistent_project/instructions_paspoort_stripped_v2.csv'
pat = data_path + 'opslag_inclusieve_spraakassistent_project/patterns_v2.csv'
qa_travel = data_path + 'opslag_inclusieve_spraakassistent_project/Vraag_antwoord_ov_v5.csv'
qa_passport = data_path + 'opslag_inclusieve_spraakassistent_project/Vraag_antwoord_paspoort.csv'
nav = data_path + 'opslag_inclusieve_spraakassistent_project/navigation.csv'

importlib.reload(instruction_agent)
assistant = instruction_agent.InstructAgent(llm)
assistant.prepare_instructions(instructions_travel,'travel')
assistant.prepare_instructions(instructions_passport,'passport')
assistant.prepare_patterns(pat)
assistant.setup_rag(collection,[['qa','travel',qa_travel],['qa','passport',qa_passport],['nav',nav]])

In [21]:
assistant.chat()

Hallo! Ik kan je helpen met 'een reis plannen' of 'een afspraak maken voor een paspoort'. Je kan 'vorige' zeggen als je een stap terug wilt doen, 'volgende' als je naar de volgende stap wilt. Zeg 'herhaal' als je wilt dat ik een stap herhaal. Je kan altijd een vraag stellen. Zeg nu 'reis' of 'paspoort'.


You:  paspoort


Agent: Ik ga je instrueren om een paspoort aan te vragen op de website van de gemeente Amsterdam. Stap 1: Scroll naar beneden. Klik op de blauwe tekst 'Paspoort, ID-kaart en rijbewijs'.


You:  gedaan


Agent: Klik op de blauwe tekst 'Paspoort 18 jaar en ouder'.


...

Agent: Klik op de knop '2 Kies een locatie'


You:  check


Agent: Bedoelde je "Okee"?


You:  ja


TypeError: InstructAgent.select_response() missing 1 required positional argument: 'do'